# COCO pre-trained vs. custom YOLOv26

This notebook compares **YOLOv26 pre-trained on COCO** with **YOLOv26 trained on the Urban Disaster Monitor custom dataset**.

- In COCO, civilian and rescuer are mapped to the "person" class
- In the Urban Disaster Monitor dataset, the classes are distinct.

Here we evaluate both models on the same images (with labels mapped to COCO IDs) and compare Precision, Recall, mAP@0.5 and mAP@0.5–0.95.

---
**Project:** [Urban Disaster Monitor](https://github.com/MariaCarolinass/urban-disaster-monitor) · **Authors:** Carolina Soares, João Galdino

## 1. Prepare dataset for COCO evaluation

The Urban Disaster Monitor dataset has 6 classes (civilian, rescuer, cat, dog, horse, cow). To evaluate the **COCO pre-trained** model, we map annotations to COCO IDs:

- civilian and rescuer → 0 (person)
- cat → 15
- dog → 16
- horse → 17
- cow → 19

Thus the pre-trained YOLO only sees "person" + animals, on equal footing with the custom model evaluation.

In [ ]:
import os
import shutil
from pathlib import Path

# Keep the dataset in YOLO format under `dataset/` (or adjust `INPUT_DATASET`)
INPUT_DATASET = "../dataset"
OUTPUT_DATASET = "../dataset_filtered"

# Mapping: original dataset class -> COCO ID
class_coco_id_mapping = {
    0: 15,  # cat (original) -> 15 (COCO ID for cat)
    1: 0,   # person/civilian (original) -> 0 (COCO ID for person)
    2: 19,  # cow (original) -> 19 (COCO ID for cow)
    3: 16,  # dog (original) -> 16 (COCO ID for dog)
    4: 17,  # horse (original) -> 17 (COCO ID for horse)
    5: 0    # person/rescuer (original) -> 0 (COCO ID for person)
}

def filter_annotations(input_dir, output_dir, split='train'):
    """Filter and reindex classes to COCO IDs."""
    labels_in = Path(input_dir) / split / 'labels'
    labels_out = Path(output_dir) / split / 'labels'
    images_in = Path(input_dir) / split / 'images'
    images_out = Path(output_dir) / split / 'images'

    # Clear output directory if it already exists
    if labels_out.exists():
        shutil.rmtree(labels_out)
    if images_out.exists():
        shutil.rmtree(images_out)

    labels_out.mkdir(parents=True, exist_ok=True)
    images_out.mkdir(parents=True, exist_ok=True)

    processed = 0
    skipped = 0

    for label_file in labels_in.glob('*.txt'):
        new_lines = []

        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue

                try:
                    old_class = int(parts[0])

                    # Remap to COCO ID if class is in mapping
                    if old_class in class_coco_id_mapping:
                        coco_class_id = class_coco_id_mapping[old_class]
                        # Keep the rest of the line (coordinates)
                        new_lines.append(f"{coco_class_id} {' '.join(parts[1:])}\n")
                except (ValueError, IndexError):
                    print(f"Malformed line in {label_file.name}: {line}")
                    continue

        # Only copy if there is at least one valid annotation
        if new_lines:
            # Save label file
            with open(labels_out / label_file.name, 'w') as f:
                f.writelines(new_lines)

            # Copy matching image
            image_copied = False
            for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
                image_path = images_in / (label_file.stem + ext)
                if image_path.exists():
                    shutil.copy(image_path, images_out / image_path.name)
                    processed += 1
                    image_copied = True
                    break

            if not image_copied:
                print(f"Warning: Label {label_file.name} has no matching image")
        else:
            skipped += 1

    print(f"  {split}: {processed} images processed, {skipped} skipped (no valid classes)")

# Remove old cache if present
cache_files = [
    f"{OUTPUT_DATASET}/train/labels.cache",
    f"{OUTPUT_DATASET}/valid/labels.cache",
    f"{OUTPUT_DATASET}/test/labels.cache",
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"Cache removed: {cache_file}")

input_dataset = INPUT_DATASET
output_dataset = OUTPUT_DATASET

if not Path(input_dataset).exists():
    print("Warning: dataset folder not found. Download the dataset (e.g. training-yolo-dataset notebook) and adjust INPUT_DATASET.")

print("\nFiltering dataset with COCO IDs...")
for split in ['train', 'valid', 'test']:
    if os.path.exists(f"{input_dataset}/{split}"):
        filter_annotations(input_dataset, output_dataset, split)

print("\nFiltered dataset created!")
print("\nChecking a sample label...")

# Check a sample label
valid_labels_dir = Path(output_dataset) / "valid" / "labels"
valid_labels = list(valid_labels_dir.glob("*.txt")) if valid_labels_dir.exists() else []
if valid_labels:
    with open(valid_labels[0], 'r') as f:
        print(f"\nContents of {valid_labels[0].name}:")
        print(f.read())
else:
    print("ERROR: No labels were created in the valid directory!")


Filtering dataset with COCO IDs...
  train: 2674 images processed, 0 skipped (no valid classes)
  valid: 383 images processed, 0 skipped (no valid classes)
  test: 183 images processed, 0 skipped (no valid classes)

Filtered dataset created!

Checking a sample label...

Contents of image_5b495164fcad4f28b5a43c7bd2e7c4b4_png.rf.24604ff8be7d559b586cfdc67eba4c8a.txt:
17 0.72109375 0.43984375 0.20078125 0.14921875
17 0.53984375 0.37109375 0.1671875 0.08125
17 0.44765625 0.50859375 0.3515625 0.221875
17 0.22265625 0.5 0.26640625 0.19140625
17 0.2609375 0.42578125 0.11171875 0.05546875



Create `data.yaml` for the filtered dataset (classes in COCO format).

In [22]:
# data.yaml for filtered dataset (evaluation on COCO classes used)
data_yaml_path = Path(output_dataset) / "data_filtered.yaml"
data_yaml_path.parent.mkdir(parents=True, exist_ok=True)
content = f"""train: ../train/images
val: ../valid/images
test: ../test/images

nc: 20 # Max COCO ID + 1 (Cow is 19)
names: [
    'person', 'null', 'null', 'null', 'null', 'null', 'null', 'null', 'null', 'null',
    'null', 'null', 'null', 'null', 'null', 'cat', 'dog', 'horse', 'null', 'cow'
]
"""
data_yaml_path.write_text(content)
print(f"Created: {data_yaml_path}")

Created: ../dataset_filtered/data_filtered.yaml


## 2. Evaluate COCO pre-trained and custom models

Install the Ultralytics library.

In [10]:
!pip install ultralytics --quiet

Run validation of pre-trained YOLOv26 on the filtered dataset.

In [25]:
# classes: 0=person, 15=cat, 16=dog, 17=horse, 19=cow
PRETRAINED_MODEL = "yolo26m.pt"
PRETRAINED_DATA_YAML = str(Path(output_dataset) / "data_filtered.yaml")
COCO_CLASSES = [0, 15, 16, 17, 19]

100%|██████████████████████████████████████| 38.8M/38.8M [00:02<00:00, 14.9MB/s]
Ultralytics 8.3.162 🚀 Python-3.12.3 torch-2.7.1+cu126 CPU (13th Gen Intel Core(TM) i7-1355U)
YOLO26m summary (fused): 125 layers, 20,091,712 parameters, 0 gradients, 68.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5601.3±1560.7 MB/s, size: 70.9 KB)
val: Scanning /home/carol/urban-disaster-monitor/dataset_filtered/valid/labels.c
/home/carol/urban-disaster-monitor/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                 Class     Images  Instances      Box(P          R      mAP50  m
                   all        383       1383      0.868      0.707      0.803      0.422
                person        259        815      0.875      0.698      0.769      0.337
                   cat         35        123      0.953      0.6

Run validation of the **custom** model on the **original** dataset.

In [ ]:
CUSTOM_MODEL = Path("../models/yolov26m/weights/best.pt")
CUSTOM_DATA_YAML = str(Path(input_dataset) / "data.yaml")
if not CUSTOM_MODEL.exists():
    raise FileNotFoundError(f"Custom checkpoint not found: {CUSTOM_MODEL}")

Ultralytics 8.3.162 🚀 Python-3.12.3 torch-2.7.1+cu126 CPU (13th Gen Intel Core(TM) i7-1355U)
YOLO26m summary (fused): 125 layers, 20,034,658 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4339.1±1444.3 MB/s, size: 65.5 KB)
val: Scanning /home/carol/urban-disaster-monitor/dataset/valid/labels... 383 ima
val: New cache created: /home/carol/urban-disaster-monitor/dataset/valid/labels.cache
/home/carol/urban-disaster-monitor/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                 Class     Images  Instances      Box(P          R      mAP50  m
                   all        383       1383      0.874      0.772      0.861      0.516
                   cat         35        123      0.909      0.789      0.883      0.545
              civilian        211        633      0.875    

## 3. Custom vs. COCO pre-trained comparison

The chart below is generated by evaluating the two checkpoints in the preceding cells. It does not use fixed metric values.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

def metric_values(validation_result):
    box = validation_result.box
    return [box.mp, box.mr, box.map50, box.map]

pretrained_result = YOLO(PRETRAINED_MODEL).val(
    data=PRETRAINED_DATA_YAML,
    imgsz=640,
    classes=COCO_CLASSES,
    verbose=False,
)
custom_result = YOLO(str(CUSTOM_MODEL)).val(data=CUSTOM_DATA_YAML, imgsz=640, verbose=False)

metrics = ["Precision", "Recall", "mAP@0.5", "mAP@0.5–0.95"]
yolo_custom = metric_values(custom_result)
yolo_pretrained = metric_values(pretrained_result)

x = np.arange(len(metrics))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4.5))

bars1 = ax.bar(x - width / 2, yolo_custom, width, label="YOLOv26m Custom", color="#2A9D8F", edgecolor="black", hatch="//")
bars2 = ax.bar(x + width / 2, yolo_pretrained, width, label="YOLOv26m Pretrained", color="#F4A261", edgecolor="black", hatch="--")

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)

ax.set_ylabel("Metric Value")
ax.set_xlabel("Performance Metrics")
ax.set_title("Custom vs Pretrained YOLOv26m - Detection Metrics")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.0)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()